# 03: 統合パイプライン（NER + RAG）

## 1. ライブラリインポート

In [1]:
import os
import torch
from dotenv import load_dotenv
load_dotenv()

from transformers import pipeline
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("インポート完了")

インポート完了


## 2. デバイス設定

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"デバイス: {device}")

デバイス: cuda


## 3. NERモデル初期化

In [3]:
ner_pipeline = pipeline(
    "token-classification",
    model="../models/biobert-ner-bc5cdr",
    tokenizer="../models/biobert-ner-bc5cdr",
    aggregation_strategy="first",
    device=0 if device == "cuda" else -1
)

print("NERモデル初期化完了")

NERモデル初期化完了


## 4. NER動作確認

In [4]:
# テスト - metforminは検出される
test_text = "Patient has diabetes and takes metformin. What are the side effects?"
entities = ner_pipeline(test_text)

print(f"テキスト: {test_text}\n")
print("抽出結果:")
for ent in entities:
    print(f"  - {ent['entity_group']}: {ent['word']}")

テキスト: Patient has diabetes and takes metformin. What are the side effects?

抽出結果:
  - Disease: diabetes
  - Chemical: metformin


## 5. RAG初期化

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': device}
)
vectorstore = Chroma(
    persist_directory="../../02-Medical-RAG/results/chroma_db",
    embedding_function=embeddings
)
llm = ChatOpenAI(
    model="GLM-4.7",
    base_url="https://api.z.ai/api/coding/paas/v4",
    api_key=os.getenv("OPENAI_API_KEY")
)

print("RAG初期化完了")

C:\Users\hello\AppData\Local\Temp\ipykernel_1564\1928502522.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


RAG初期化完了


## 6. 統合パイプライン関数

In [6]:
def integrated_qa(question):
    """NER + RAG 統合パイプライン"""
    # NER
    entities = ner_pipeline(question)
    
    # すべてのエンティティをキーワードとして使用
    keywords = [ent['word'] for ent in entities]
    
    # 検索クエリ拡張
    search_query = f"{question} {' '.join(keywords)}" if keywords else question
    
    # 検索
    documents = vectorstore.similarity_search(search_query, k=5)
    context = "\n\n".join([f"[{i}] {doc.page_content.strip()}" for i, doc in enumerate(documents, 1)])
    
    # 回答生成
    prompt = f"""Context: {context}
Question: {question}
Answer:"""
    response = llm.invoke(prompt)
    
    return response.content, entities, search_query

print("統合パイプライン関数定義完了")

統合パイプライン関数定義完了


## 7. デモ実行

In [7]:
# question = "Patient has diabetes and takes metformin. What are the side effects?"
question = "Patient has diabetes and takes insulin. What are the side effects?"

answer, entities, search_query = integrated_qa(question)

print(f"Q: {question}\n")
print(f"抽出エンティティ:")
for ent in entities:
    print(f"  - {ent['entity_group']}: {ent['word']}")
print(f"\n検索クエリ: {search_query}\n")
print(f"A: {answer}")

Q: Patient has diabetes and takes insulin. What are the side effects?

抽出エンティティ:
  - Disease: diabetes
  - Chemical: insulin

検索クエリ: Patient has diabetes and takes insulin. What are the side effects? diabetes insulin

A: The provided context does not contain information about the side effects of taking insulin. The text only lists the health problems caused by diabetes itself (such as heart disease, stroke, nerve damage, kidney issues, gum disease, vision loss, and foot problems).


## 8. GLM-4評価　NLPの結果を簡易的に外部APIで評価

In [8]:
eval_prompt = f"""以下の統合医療Q&Aシステムを評価してください。

質問: {question}
抽出エンティティ: {[f"{e['entity_group']}:{e['word']}" for e in entities]}
検索クエリ: {search_query}
回答: {answer}

まず以下を日本語で翻訳して表示してください：
【翻訳】
Q: (日本語訳)
A: (日本語訳)

次に採点してください（各100点満点）：
1. エンティティ抽出 (25点)
2. 検索クエリ (25点)
3. 回答品質 (50点)

重要ルール：「情報がありません」は0-30点、医学的な具体的情報が正しい場合は70-100点

結果：
- 総合点: (0-100)
- フィードバック: (1文)"""

eval_response = llm.invoke(eval_prompt)
print(eval_response.content)

【翻訳】
Q: 患者は糖尿病でインスリンを使用しています。副作用は何ですか？
A: 提供された文脈（コンテキスト）には、インスリン服用による副作用に関する情報が含まれていません。そのテキストには、糖尿病自体によって引き起こされる健康問題（心臓病、脳卒中、神経障害、腎臓の問題、歯肉病、視力低下、足の問題など）のみが記載されています。

次に採点してください（各100点満点）：

1. **エンティティ抽出**: 25点
   - 理由: "diabetes"を疾病、"insulin"を化学物質として正確に抽出できており、過不足もありません。
2. **検索クエリ**: 25点
   - 理由: 質問文全体に加え、抽出した重要キーワード（diabetes insulin）を適切に組み合わせており、検索意図を明確に表しています。
3. **回答品質**: 30点
   - 理由: 検索コンテキストに情報がないことを正確に述べており、ハルシネーション（嘘）はありません。しかし、重要ルールにある通り「情報がありません」に該当するため、得点は30点とします。

**結果**:
- **総合点**: 80
- **フィードバック**: エンティティ抽出と検索クエリの生成は完璧ですが、肝心な医学情報を検索できておらず、質問に答えることはできませんでした。
